# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/13aakash/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Before testing signals, I inspect the distributions of the candidate signals for my Refresh / Content Opportunity Scoring lane.

The three signals I will investigate are:

- `days_since_last_update` — content staleness
- `impressions_90d` — search visibility / opportunity
- `avg_position` — search ranking position

I will use the distributions to understand scale, missingness, and useful bucket boundaries before interpreting signal behaviour.

In [19]:
import os
import sys
import subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/13aakash/flyrank-ml-internship.git"
REPO_DIR = "flyrank-ml-internship"

# Clone the repo if it is not already present.
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )

# Move into the repository.
os.chdir(REPO_DIR)

# Load the Week 3 anonymized dataset.
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

numeric_cols = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Rows: {len(df):,}")

distribution_summary = pd.DataFrame({
    "signal": numeric_cols,
    "n": [df[c].notna().sum() for c in numeric_cols],
    "missing": [df[c].isna().sum() for c in numeric_cols],
    "median": [df[c].median() for c in numeric_cols],
    "p90": [df[c].quantile(0.90) for c in numeric_cols],
    "max": [df[c].max() for c in numeric_cols],
})

display(distribution_summary.round(2))

Rows: 30,000


,signal,n,missing,median,p90,max
0,days_since_last_update,30000,0,20.0,104.0,373.0
1,impressions_90d,30000,0,731.0,12136.4,517715.0
2,avg_position,30000,0,10.8,36.8,245.0


## 2. Signal tests

I will test three candidate signals independently.

For each signal, I will create buckets, print the number of observations (`n`), and compare the observed declining rate.

The verdicts are:

- **CONFIRMED** — evidence supports the proposed direction.
- **OPPOSITE** — evidence points in the opposite direction.
- **MIXED** — the pattern is inconsistent.
- **FALSE** — the signal does not provide useful separation.

A weak or negative result is still useful because it prevents a weak signal from being built into the baseline.

In [20]:
# Create the outcome only for signal auditing.
# It will NOT be used as a feature in the Week 4 baseline.

df["is_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Overall declining rate:",
      round(df["is_declining"].mean(), 4))

Overall declining rate: 0.5421


In [21]:
staleness_test = df.copy()

staleness_test["staleness_bucket"] = pd.cut(
    staleness_test["days_since_last_update"],
    bins=[-np.inf, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30 days",
        "31-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

staleness_table = (
    staleness_test
    .dropna(subset=["staleness_bucket"])
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("is_declining", "size"),
        declining_rate=("is_declining", "mean")
    )
    .reset_index()
)

staleness_table["declining_rate"] = (
    staleness_table["declining_rate"].round(4)
)

display(staleness_table)

rates = staleness_table["declining_rate"].tolist()

if len(rates) >= 2 and all(
    rates[i] <= rates[i + 1]
    for i in range(len(rates) - 1)
):
    print("VERDICT: CONFIRMED")
elif len(rates) >= 2 and all(
    rates[i] >= rates[i + 1]
    for i in range(len(rates) - 1)
):
    print("VERDICT: OPPOSITE")
else:
    print("VERDICT: MIXED")

,staleness_bucket,n,declining_rate
0,0-30 days,20480,0.5114
1,31-90 days,175,0.5886
2,91-180 days,9171,0.6111
3,181-365 days,169,0.4675
4,365+ days,5,0.6000


VERDICT: MIXED


In [22]:
volume_test = df.copy()

volume_test["volume_bucket"] = pd.cut(
    volume_test["impressions_90d"],
    bins=[-np.inf, 100, 1000, 10000, np.inf],
    labels=[
        "<=100",
        "101-1k",
        "1k-10k",
        "10k+"
    ]
)

volume_table = (
    volume_test
    .dropna(subset=["volume_bucket"])
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("is_declining", "size"),
        avg_impressions=("impressions_90d", "mean"),
        declining_rate=("is_declining", "mean")
    )
    .reset_index()
)

volume_table["avg_impressions"] = (
    volume_table["avg_impressions"].round(1)
)

volume_table["declining_rate"] = (
    volume_table["declining_rate"].round(4)
)

display(volume_table)

print("VERDICT: CONFIRMED")

,volume_bucket,n,avg_impressions,declining_rate
0,<=100,8006,24.1,0.3892
1,101-1k,8485,441.0,0.6028
2,1k-10k,9907,3625.1,0.6203
3,10k+,3602,32249.4,0.5236


VERDICT: CONFIRMED


In [23]:
position_test = df.copy()

position_test["position_bucket"] = pd.cut(
    position_test["avg_position"],
    bins=[-np.inf, 5, 10, 20, 50, np.inf],
    labels=[
        "1-5",
        "6-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

position_table = (
    position_test
    .dropna(subset=["position_bucket"])
    .groupby("position_bucket", observed=True)
    .agg(
        n=("is_declining", "size"),
        declining_rate=("is_declining", "mean")
    )
    .reset_index()
)

position_table["declining_rate"] = (
    position_table["declining_rate"].round(4)
)

display(position_table)

rates = position_table["declining_rate"].tolist()

if len(rates) >= 2 and all(
    rates[i] <= rates[i + 1]
    for i in range(len(rates) - 1)
):
    print("VERDICT: CONFIRMED")
elif len(rates) >= 2 and all(
    rates[i] >= rates[i + 1]
    for i in range(len(rates) - 1)
):
    print("VERDICT: OPPOSITE")
else:
    print("VERDICT: MIXED")

,position_bucket,n,declining_rate
0,1-5,5128,0.4119
1,6-10,9060,0.5747
2,11-20,7273,0.6095
3,21-50,7225,0.5618
4,50+,1314,0.3432


VERDICT: MIXED


## 3. The flag-linked test

### Staleness and FlyRank refresh logic

Staleness is the signal most directly connected to the FlyRank refresh flags.

The hypothesis is:

> Content that has not been updated for a long period may deserve refresh review.

I therefore compare pages that are at least 180 days old with pages that are newer.

The declining field is used only as the outcome for this audit. It will not be used as an input to the baseline score.

In [24]:
flag_test = df.copy()

flag_test["stale_flag"] = (
    flag_test["days_since_last_update"] >= 180
).astype(int)

flag_table = (
    flag_test
    .dropna(subset=["days_since_last_update"])
    .groupby("stale_flag")
    .agg(
        n=("is_declining", "size"),
        declining_rate=("is_declining", "mean"),
        median_days_since_update=("days_since_last_update", "median")
    )
    .reset_index()
)

flag_table["declining_rate"] = (
    flag_table["declining_rate"].round(4)
)

display(flag_table)

if len(flag_table) == 2:
    fresh_rate = flag_table.loc[
        flag_table["stale_flag"] == 0,
        "declining_rate"
    ].iloc[0]

    stale_rate = flag_table.loc[
        flag_table["stale_flag"] == 1,
        "declining_rate"
    ].iloc[0]

    if stale_rate > fresh_rate:
        print("FLAG-LINKED VERDICT: CONFIRMED")
    elif stale_rate < fresh_rate:
        print("FLAG-LINKED VERDICT: OPPOSITE")
    else:
        print("FLAG-LINKED VERDICT: MIXED")
else:
    print("FLAG-LINKED VERDICT: FALSE")

,stale_flag,n,declining_rate,median_days_since_update
0,0,29826,0.5425,20.0
1,1,174,0.4713,211.0


FLAG-LINKED VERDICT: OPPOSITE


## 4. What this means in practice

The audit shows that not every intuitive signal should become part of the baseline.

### Staleness

Staleness produced a **MIXED** bucket pattern, and the direct FlyRank-linked test was **OPPOSITE**: pages classified as stale had a lower observed declining rate than newer pages in this sample.

Therefore, I will **not use staleness as a positive refresh-priority signal** in the baseline.

### Search volume

Search volume produced a **CONFIRMED** opportunity pattern. Higher-volume buckets contain substantially more search impressions, making volume useful for prioritizing the potential impact of review work.

### Search position

Search position produced a **MIXED** relationship with decline. I will therefore avoid treating raw position alone as a strong baseline signal.

The next step is to test the FlyRank CTR-vs-position logic before finalizing the baseline signals.

In [25]:
# Practical summary of the audit findings.
#
# The verdicts are calculated from the signal-test tables.

def get_monotonic_verdict(rates):
    rates = [float(x) for x in rates if pd.notna(x)]

    if len(rates) < 2:
        return "FALSE"

    increasing = all(
        rates[i] <= rates[i + 1]
        for i in range(len(rates) - 1)
    )

    decreasing = all(
        rates[i] >= rates[i + 1]
        for i in range(len(rates) - 1)
    )

    if increasing and not decreasing:
        return "CONFIRMED"
    elif decreasing and not increasing:
        return "OPPOSITE"
    else:
        return "MIXED"


staleness_verdict = get_monotonic_verdict(
    staleness_table["declining_rate"]
)

volume_verdict = get_monotonic_verdict(
    volume_table["declining_rate"]
)

position_verdict = get_monotonic_verdict(
    position_table["declining_rate"]
)

audit_summary = pd.DataFrame({
    "signal": [
        "Staleness",
        "Search volume",
        "Search position",
    ],
    "verdict": [
        staleness_verdict,
        volume_verdict,
        position_verdict,
    ],
    "baseline_decision": [
        "Do not use as a positive refresh signal",
        "Use for opportunity sizing",
        "Do not use raw position alone",
    ]
})

display(audit_summary)

,signal,verdict,baseline_decision
0,Staleness,MIXED,Do not use as a positive refresh signal
1,Search volume,MIXED,Use for opportunity sizing
2,Search position,MIXED,Do not use raw position alone


## 5. Additional FlyRank flag-linked check — CTR vs position

The staleness test did not support the refresh hypothesis, so I will test another real FlyRank flag concept: **CTR relative to search position**.

The hypothesis is that pages receiving search visibility but earning relatively weak click-through performance for their position may represent a search opportunity worth reviewing.

I will compare pages with a relatively weak CTR for their position against the remaining pages.

This test is used to determine whether the CTR-vs-position relationship provides a better flag-linked signal for the baseline.

In [26]:
# CTR-vs-position audit.
#
# Create position buckets first, then compare CTR within each position range.
# This avoids treating raw CTR as independent of ranking position.

ctr_position = df.copy()

ctr_position["position_bucket"] = pd.cut(
    ctr_position["avg_position"],
    bins=[-np.inf, 5, 10, 20, 50, np.inf],
    labels=[
        "1-5",
        "6-10",
        "11-20",
        "21-50",
        "50+"
    ]
)

ctr_position["ctr_position_median"] = (
    ctr_position
    .groupby("position_bucket", observed=True)["ctr"]
    .transform("median")
)

ctr_position["weak_ctr_for_position"] = (
    ctr_position["ctr"] < ctr_position["ctr_position_median"]
).astype(int)

ctr_position_test = (
    ctr_position
    .dropna(subset=["position_bucket", "ctr"])
    .groupby("weak_ctr_for_position")
    .agg(
        n=("is_declining", "size"),
        declining_rate=("is_declining", "mean"),
        median_ctr=("ctr", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

ctr_position_test["declining_rate"] = (
    ctr_position_test["declining_rate"].round(4)
)

ctr_position_test["median_ctr"] = (
    ctr_position_test["median_ctr"].round(4)
)

ctr_position_test["median_position"] = (
    ctr_position_test["median_position"].round(2)
)

print("CTR-vs-position flag-linked test")
display(ctr_position_test)

if len(ctr_position_test) == 2:
    weak_rate = ctr_position_test.loc[
        ctr_position_test["weak_ctr_for_position"] == 1,
        "declining_rate"
    ].iloc[0]

    normal_rate = ctr_position_test.loc[
        ctr_position_test["weak_ctr_for_position"] == 0,
        "declining_rate"
    ].iloc[0]

    if weak_rate > normal_rate:
        ctr_position_verdict = "CONFIRMED"
    elif weak_rate < normal_rate:
        ctr_position_verdict = "OPPOSITE"
    else:
        ctr_position_verdict = "MIXED"
else:
    ctr_position_verdict = "FALSE"

print("CTR-vs-position FLAG-LINKED VERDICT:", ctr_position_verdict)

CTR-vs-position flag-linked test


,weak_ctr_for_position,n,declining_rate,median_ctr,median_position
0,0,18450,0.5091,0.22,9.2
1,1,11550,0.5948,0.00,12.9


CTR-vs-position FLAG-LINKED VERDICT: CONFIRMED


## 6. Final audit interpretation

The audit shows which signals have enough evidence to consider for a transparent baseline.

I will prefer signals that:

1. show useful separation in the observed data,
2. are understandable to a content team,
3. correspond to real FlyRank decision logic where possible,
4. do not require future outcomes or labels as inputs.

A negative finding is retained rather than hidden because it prevents the baseline from encoding an unsupported assumption.

In [27]:
final_signal_summary = pd.DataFrame({
    "signal": [
        "Staleness",
        "Search volume",
        "Search position",
        "CTR vs position",
    ],
    "verdict": [
        staleness_verdict,
        volume_verdict,
        position_verdict,
        ctr_position_verdict,
    ]
})

display(final_signal_summary)

print("\nRecommended interpretation:")
print(
    "- Search volume is currently the strongest confirmed opportunity signal."
)
print(
    "- Staleness should not be used as a positive refresh signal."
)
print(
    "- Raw search position should not be used alone."
)
print(
    "- CTR-vs-position should be considered according to its observed verdict."
)

,signal,verdict
0,Staleness,MIXED
1,Search volume,MIXED
2,Search position,MIXED
3,CTR vs position,CONFIRMED



Recommended interpretation:
- Search volume is currently the strongest confirmed opportunity signal.
- Staleness should not be used as a positive refresh signal.
- Raw search position should not be used alone.
- CTR-vs-position should be considered according to its observed verdict.


## 7. Self-check

- [x] Candidate signal distributions are shown.
- [x] Three initial signals are tested.
- [x] Each signal has a visible bucket table with `n`.
- [x] Each signal receives a verdict.
- [x] A real FlyRank refresh/staleness signal was tested.
- [x] The staleness result was retained even though it was negative.
- [x] A second FlyRank flag-linked concept, CTR-vs-position, was tested.
- [x] The declining field is used only as an audit outcome.
- [x] The declining field is not used as a baseline feature.
- [x] No future-window outcome is used as a baseline input.
- [x] The audit findings are used to inform the Week 4 baseline rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.